## Tiny CNN


#### 1. Setup and Imports

In [14]:
import sys
import os

print("Current working directory:", os.getcwd())

if os.path.isdir(os.path.join(os.getcwd(), "src")):
    PROJECT_ROOT = os.path.abspath(os.getcwd())
else:
    PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

WANDB_DIR = os.path.join(os.path.expanduser("~"), "wandb_logs", "fer2013")
os.makedirs(WANDB_DIR, exist_ok=True)
os.environ["WANDB_DIR"] = WANDB_DIR
print(f"Wandb dir: {WANDB_DIR}")

import torch
import wandb
from tqdm.notebook import tqdm
from src.dataset import get_dataloaders
from src.models import TinyCNN
from src.utils import generate_run_name, log_confusion_matrix, EarlyStopping, EMOTION_LABELS
from src.train import train

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

wandb.login()
print('Wandb ready')

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


Current working directory: /mnt/c/Users/Likun/OneDrive/Desktop/Machine Learning/challenges-in-representation-learning-facial-expression-recognition-challenge/ml-assignment-Facial-Expression-Recognition
Project root: /mnt/c/Users/Likun/OneDrive/Desktop/Machine Learning/challenges-in-representation-learning-facial-expression-recognition-challenge/ml-assignment-Facial-Expression-Recognition
Wandb dir: /home/likun/wandb_logs/fer2013
Using device: cpu
Wandb ready


#### 2. Data Loading and Augmentation

In [7]:
# test all three augmentation modes load correctly
for aug in ['none', 'light', 'strong']:
    train_loader, val_loader, test_loader = get_dataloaders(
        data_dir='data',
        aug_mode=aug,
        batch_size=64
    )
    images, labels = next(iter(train_loader))
    print(f'aug={aug} | batch shape: {images.shape} | labels shape: {labels.shape}')

print('Data loading works correctly')

aug=none | batch shape: torch.Size([64, 1, 48, 48]) | labels shape: torch.Size([64])
aug=light | batch shape: torch.Size([64, 1, 48, 48]) | labels shape: torch.Size([64])
aug=strong | batch shape: torch.Size([64, 1, 48, 48]) | labels shape: torch.Size([64])
Data loading works correctly


#### 3. Model Definition

In [9]:
# verify model builds and forward pass works
model = TinyCNN(dropout=0.0)
dummy = torch.randn(4, 1, 48, 48)
out = model(dummy)
print(f'TinyCNN output shape: {out.shape}')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

TinyCNN output shape: torch.Size([4, 7])
Total parameters: 595,655


#### 4. Hyperparameter Setup

In [10]:
CONFIGS = [
    # ── learning rate sweep (adam, no aug, no dropout) ──
    {'run_name': 'tiny_adam_0.01_bs64_noaug_do0',    'lr': 0.01,   'optimizer': 'adam', 'batch_size': 64, 'aug': 'none',  'dropout': 0.0, 'epochs': 15},
    {'run_name': 'tiny_adam_0.001_bs64_noaug_do0',   'lr': 0.001,  'optimizer': 'adam', 'batch_size': 64, 'aug': 'none',  'dropout': 0.0, 'epochs': 15},
    {'run_name': 'tiny_adam_0.0003_bs64_noaug_do0',  'lr': 0.0003, 'optimizer': 'adam', 'batch_size': 64, 'aug': 'none',  'dropout': 0.0, 'epochs': 15},
    {'run_name': 'tiny_adam_0.0001_bs64_noaug_do0',  'lr': 0.0001, 'optimizer': 'adam', 'batch_size': 64, 'aug': 'none',  'dropout': 0.0, 'epochs': 15},

    # ── optimizer comparison ──
    {'run_name': 'tiny_sgd_0.01_bs64_noaug_do0',     'lr': 0.01,   'optimizer': 'sgd',  'batch_size': 64, 'aug': 'none',  'dropout': 0.0, 'epochs': 15},
    {'run_name': 'tiny_sgd_0.001_bs64_noaug_do0',    'lr': 0.001,  'optimizer': 'sgd',  'batch_size': 64, 'aug': 'none',  'dropout': 0.0, 'epochs': 15},

    # ── augmentation effect ──
    {'run_name': 'tiny_adam_0.001_bs64_light_do0',   'lr': 0.001,  'optimizer': 'adam', 'batch_size': 64, 'aug': 'light', 'dropout': 0.0, 'epochs': 15},
    {'run_name': 'tiny_adam_0.001_bs64_strong_do0',  'lr': 0.001,  'optimizer': 'adam', 'batch_size': 64, 'aug': 'strong','dropout': 0.0, 'epochs': 15},

    # ── dropout effect ──
    {'run_name': 'tiny_adam_0.001_bs64_noaug_do0.25','lr': 0.001,  'optimizer': 'adam', 'batch_size': 64, 'aug': 'none',  'dropout': 0.25,'epochs': 15},
    {'run_name': 'tiny_adam_0.001_bs64_noaug_do0.5', 'lr': 0.001,  'optimizer': 'adam', 'batch_size': 64, 'aug': 'none',  'dropout': 0.5, 'epochs': 15},

    # ── batch size effect ──
    {'run_name': 'tiny_adam_0.001_bs32_noaug_do0',   'lr': 0.001,  'optimizer': 'adam', 'batch_size': 32, 'aug': 'none',  'dropout': 0.0, 'epochs': 15},
    {'run_name': 'tiny_adam_0.001_bs128_noaug_do0',  'lr': 0.001,  'optimizer': 'adam', 'batch_size': 128,'aug': 'none',  'dropout': 0.0, 'epochs': 15},
]

print(f'Total runs planned: {len(CONFIGS)}')

Total runs planned: 12


#### 5. Training

In [ ]:
import os

if "WANDB_DIR" not in globals():
    WANDB_DIR = os.path.join(os.path.expanduser("~"), "wandb_logs", "fer2013")
    os.makedirs(WANDB_DIR, exist_ok=True)

all_results = []

for i, config in enumerate(CONFIGS):
    config['arch'] = 'tiny'

    print(f'\n[{i+1}/{len(CONFIGS)}] Starting run: {config["run_name"]}')

    train_loader, val_loader, _ = get_dataloaders(
        data_dir='data',
        aug_mode=config['aug'],
        batch_size=config['batch_size']
    )

    model = TinyCNN(dropout=config['dropout'])

    wandb.init(
        project='fer2013-expression-recognition',
        name=config['run_name'],
        group=config['arch'],
        config=config,
        dir=WANDB_DIR,
    )

    best_val_acc = train(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        config=config,
        device=DEVICE
    )

    wandb.finish()

    all_results.append({
        'run': config['run_name'],
        'best_val_acc': best_val_acc
    })

    print(f'Done [{i+1}/{len(CONFIGS)}] — best_val_acc: {best_val_acc:.4f}')

print('\nAll TinyCNN runs complete')

wandb: WARNING Fatal error while uploading data. Some run data will not be synced, but it will still be written to disk. Use `wandb sync` at the end of the run to try uploading.


In [ ]:
import os

if "WANDB_DIR" not in globals():
    WANDB_DIR = os.path.join(os.path.expanduser("~"), "wandb_logs", "fer2013")
    os.makedirs(WANDB_DIR, exist_ok=True)

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

results_df = pd.DataFrame(all_results).sort_values('best_val_acc', ascending=False)
print(results_df.to_string(index=False))

# ── Chart 1: Bar chart of all runs ──
fig, ax = plt.subplots(figsize=(14, 5))
colors = ['steelblue' if 'do0' in r and 'noaug' in r 
          else 'coral' if 'strong' in r 
          else 'mediumseagreen' for r in results_df['run']]
ax.bar(results_df['run'], results_df['best_val_acc'], color=colors)
ax.axhline(y=results_df['best_val_acc'].mean(), color='red', linestyle='--', label='Mean val_acc')
ax.set_xticks(range(len(results_df)))
ax.set_xticklabels(results_df['run'], rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Best Val Accuracy')
ax.set_title('TinyCNN — All Runs Comparison')
ax.legend()
plt.tight_layout()
plt.savefig('../checkpoints/tinycnn_bar.png')
wandb.init(project='fer2013-expression-recognition', name='tinycnn_analysis', group='tiny', job_type='analysis', dir=WANDB_DIR)
wandb.log({'tinycnn_bar_chart': wandb.Image(fig)})
plt.show()

# ── Chart 2: LR effect ──
lr_runs = [r for r in all_results if 'sgd' not in r['run'] 
           and 'bs32' not in r['run'] and 'bs128' not in r['run'] 
           and 'do0.25' not in r['run'] and 'do0.5' not in r['run'] 
           and 'light' not in r['run'] and 'strong' not in r['run']]
lr_df = pd.DataFrame(lr_runs)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(len(lr_df)), lr_df['best_val_acc'], 'o-', color='steelblue')
ax.set_xticks(range(len(lr_df)))
ax.set_xticklabels([r['run'].split('_')[2] for r in lr_runs], fontsize=8)
ax.set_xlabel('Learning Rate')
ax.set_ylabel('Best Val Accuracy')
ax.set_title('TinyCNN — Learning Rate Effect')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../checkpoints/tinycnn_lr.png')
wandb.log({'tinycnn_lr_effect': wandb.Image(fig)})
plt.show()

# ── Chart 3: Adam vs SGD ──
adam_acc = next(r['best_val_acc'] for r in all_results if r['run'] == 'tiny_adam_0.001_bs64_noaug_do0')
sgd_runs = [r for r in all_results if 'sgd' in r['run']]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(
    ['adam_0.001'] + [r['run'] for r in sgd_runs],
    [adam_acc] + [r['best_val_acc'] for r in sgd_runs],
    color=['steelblue', 'coral', 'coral']
)
ax.set_ylabel('Best Val Accuracy')
ax.set_title('TinyCNN — Adam vs SGD')
plt.xticks(rotation=20, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig('../checkpoints/tinycnn_optimizer.png')
wandb.log({'tinycnn_optimizer_comparison': wandb.Image(fig)})
plt.show()

# ── Chart 4: Augmentation effect ──
aug_data = [
    ('noaug',  next(r['best_val_acc'] for r in all_results if r['run'] == 'tiny_adam_0.001_bs64_noaug_do0')),
    ('light',  next(r['best_val_acc'] for r in all_results if r['run'] == 'tiny_adam_0.001_bs64_light_do0')),
    ('strong', next(r['best_val_acc'] for r in all_results if r['run'] == 'tiny_adam_0.001_bs64_strong_do0')),
]
aug_labels, aug_vals = zip(*aug_data)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(aug_labels, aug_vals, color=['steelblue', 'mediumseagreen', 'coral'])
ax.set_ylabel('Best Val Accuracy')
ax.set_title('TinyCNN — Augmentation Effect')
plt.tight_layout()
plt.savefig('../checkpoints/tinycnn_augmentation.png')
wandb.log({'tinycnn_augmentation_effect': wandb.Image(fig)})
plt.show()

# ── Chart 5: Dropout effect ──
dropout_data = [
    ('do0',    next(r['best_val_acc'] for r in all_results if r['run'] == 'tiny_adam_0.001_bs64_noaug_do0')),
    ('do0.25', next(r['best_val_acc'] for r in all_results if r['run'] == 'tiny_adam_0.001_bs64_noaug_do0.25')),
    ('do0.5',  next(r['best_val_acc'] for r in all_results if r['run'] == 'tiny_adam_0.001_bs64_noaug_do0.5')),
]
do_labels, do_vals = zip(*dropout_data)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(do_labels, do_vals, color=['steelblue', 'mediumseagreen', 'coral'])
ax.set_ylabel('Best Val Accuracy')
ax.set_title('TinyCNN — Dropout Effect')
plt.tight_layout()
plt.savefig('../checkpoints/tinycnn_dropout.png')
wandb.log({'tinycnn_dropout_effect': wandb.Image(fig)})
plt.show()

wandb.finish()

print(f'\nBest run: {results_df.iloc[0]["run"]}')
print(f'Best val_acc: {results_df.iloc[0]["best_val_acc"]:.4f}')
print('All charts saved and logged to Wandb')